## Játék

In [ ]:
import numpy as np
import time
import random
from IPython.display import clear_output

PosMin = 0
PosMax = 8
Height = 20
PPos = 4
Enemy = []
Spawnchance = 20
Score = 0
GameOver = False

def CreateEnemy(X):
  Enemy.append([0, X])

def RandomSpawn():
  global Spawnchance
  rnd = random.randint(1, 100)
  if (rnd < Spawnchance): CreateEnemy(random.randint(PosMin, PosMax - 1))
  if random.randint(1, 2) == 1: Spawnchance = Spawnchance + 1

def MoveLeft():
  global PPos
  if PPos > PosMin: PPos = PPos - 1

def MoveRight():
  global PPos
  if PPos < PosMax - 1: PPos = PPos + 1

def Defeat():
  global GameOver
  GameOver = True

def Reset():
  global PPos
  global Enemy
  global GameOver
  global Score
  global Spawnchance
  PPos = 4
  Enemy = []
  GameOver = False
  Score = 0
  Spawnchance = 20

def NextStep():
  global Score
  Score = Score + 1
  for E in Enemy:
    E[0] = E[0] + 1
    if E[0] == Height and E[1] == PPos: Defeat()
    if E[0] > Height: Enemy.remove(E)

def Draw():
  clear_output()
  map = np.zeros([Height + 1, PosMax])
  for E in Enemy:
    map[E[0]][E[1]] = 2
  map[Height][PPos] = 1

  for i in map:
    print('| ', end = '')
    for c in i:
      if c == 0: print('  ', end = '')
      if c == 1: print('A ', end = '')
      if c == 2: print('# ', end = '')
    print('|', end = '')
    print()
  print(Score)

def GameSeed(Sleep, Skip, Seed):
  Reset()
  Action = 0
  random.seed(Seed)

  while(not GameOver):
    for i in range(Skip + 1):
      time.sleep(Sleep)
      Sense = AISense()
      Action = AIThink(Sense[0], Sense[1], Sense[2], Brain)
      if Action == -1: MoveLeft()
      if Action == 1: MoveRight()
      NextStep()
      RandomSpawn()
      if GameOver: break
    Draw()

def Game(Sleep, Skip):
  Reset()
  Action = 0

  while(not GameOver):
    for i in range(Skip + 1):
      time.sleep(Sleep)
      Sense = AISense()
      Action = AIThink(Sense[0], Sense[1], Sense[2], Brain)
      if Action == -1: MoveLeft()
      if Action == 1: MoveRight()
      NextStep()
      RandomSpawn()
      if GameOver: break
    Draw()

In [ ]:
Game(0.5, 0)

|               # |
|           #     |
|     #           |
|   #             |
| #               |
|         #       |
|         #       |
| #               |
|     #           |
|             #   |
|   #             |
| #               |
|               # |
|           #     |
|             #   |
| #               |
|               # |
|       #         |
|               # |
|             #   |
|       #       A |
157


In [ ]:
GameSeed(0.3, 0, 69)

|     #           |
|         #       |
|           #     |
|   #             |
|               # |
|         #       |
|   #             |
|       #         |
| #               |
|         #       |
| #               |
|         #       |
|               # |
|               # |
|             #   |
|           #     |
|     #           |
|             #   |
|   #             |
|   #             |
|             # A |
201


## AI

In [ ]:
Brain = np.zeros([8, 3])

In [ ]:
Brain = [[-2.31, -8.9,   5.87],
        [-8.18,  0.37,  4.63],
        [-4.09, -4.23,  3.64],
        [ 5.98,  2.83, -4.94],
        [-1.59,  5.55, -0.69],
        [-6.81, -4.15,  4.87],
        [-0.86,  0.5,  -1.07],
        [ 0.52, -0.17,  0.59]]

In [ ]:
Brain = [[ -10.42, -100.05,   78.66],
        [ -31.41,   -8.37,   44.29],
        [ -71.26,  -11.54,  -24.47],
        [  78.61,   14.53,  -68.74],
        [ -59.16,   83.37,   28.44],
        [-123.27,   -4.89,  -27.19],
        [ -30.11,   -5.66,   47.88],
        [  36.04,   59.63,   81.46]]

In [ ]:
def NeuronOutput(x):
  return max(0, x)

def AIThink(Xl, Xc, Xr, BrainLocal):
  neuron1 = NeuronOutput((Xl * BrainLocal[0][0]) + (Xc * BrainLocal[0][1]) + (Xr * BrainLocal[0][2]) + BrainLocal[3][0])
  neuron2 = NeuronOutput((Xl * BrainLocal[1][0]) + (Xc * BrainLocal[1][1]) + (Xr * BrainLocal[1][2]) + BrainLocal[3][1])
  neuron3 = NeuronOutput((Xl * BrainLocal[2][0]) + (Xc * BrainLocal[2][1]) + (Xr * BrainLocal[2][2]) + BrainLocal[3][2])

  neuron4 = NeuronOutput((neuron1 * BrainLocal[4][0]) + (neuron2 * BrainLocal[4][1]) + (neuron3 * BrainLocal[4][2]) + BrainLocal[7][0])
  neuron5 = NeuronOutput((neuron1 * BrainLocal[5][0]) + (neuron2 * BrainLocal[5][1]) + (neuron3 * BrainLocal[5][2]) + BrainLocal[7][1])
  neuron6 = NeuronOutput((neuron1 * BrainLocal[6][0]) + (neuron2 * BrainLocal[6][1]) + (neuron3 * BrainLocal[6][2]) + BrainLocal[7][2])

  if neuron4 > neuron5 and neuron4 > neuron6: return -1
  if neuron6 > neuron5: return 1
  return 0

def AISense():
  Senses = [0, 0, 0]
  for E in Enemy:
    if E[1] == PPos - 1 and E[0] > Senses[0]: Senses[0] = E[0]
    if E[1] == PPos and E[0] > Senses[1]: Senses[1] = E[0]
    if E[1] == PPos + 1 and E[0] > Senses[2]: Senses[2] = E[0]
  Senses[0] = Senses[0] / 10
  Senses[1] = Senses[1] / 10
  Senses[2] = Senses[2] / 10
  if PPos == PosMin: Senses[0] = 5
  if PPos == PosMax: Senses[2] = 5
  return Senses

## Training

In [ ]:
from IPython.utils.text import string
def Test(BrainLocal, Seed):
  random.seed(Seed)
  Reset()
  while(not GameOver):
    NextStep()
    RandomSpawn()
    Sense = AISense()
    Action = AIThink(Sense[0], Sense[1], Sense[2], BrainLocal)
    if Action == -1: MoveLeft()
    if Action == 1: MoveRight()
  return Score

def Mutate(BrainLocal):
  NoC = 15
  Offspring = []

  BrainCopy = np.zeros([8, 3])
  for x in range(8):
    for y in range(3):
      BrainCopy[x][y] = BrainLocal[x][y]
  Offspring.append(BrainCopy)

  for i in range(NoC - 1):
    BrainCopy = np.zeros([8, 3])
    for x in range(8):
      for y in range(3):
        BrainCopy[x][y] = BrainLocal[x][y]
        if random.randint(1, 3) == 1: BrainCopy[x][y] = BrainCopy[x][y] + (random.randint(-100, 100) / 100)
    Offspring.append(BrainCopy)
  return Offspring

def TrainingEpoch(Current):
  Seed = random.randint(0, 9999)
  Subjects = Mutate(Current)
  HighScore = 0
  Leader = 0
  for i in range(len(Subjects)):
    TestScore = Test(Subjects[i], Seed)
    if TestScore >= HighScore:
      HighScore = TestScore
      Leader = i
  return Subjects[Leader]

def TrainingTotal(BrainLocal, Sessions):
  CurrentBrain = np.zeros([8, 3])
  for x in range(8):
    for y in range(3):
      CurrentBrain[x][y] = BrainLocal[x][y]
  for i in range(Sessions):
    tmp = TrainingEpoch(CurrentBrain)
    for x in range(8):
      for y in range(3):
        CurrentBrain[x][y] = tmp[x][y]
    CurrentBrain = TrainingEpoch(CurrentBrain)
    clear_output()
    print(f"{i} / {Sessions}, {i / Sessions * 100}%")

  for x in range(8):
      for y in range(3):
        BrainLocal[x][y] = CurrentBrain[x][y]

In [ ]:
TrainingTotal(Brain, 400)
Brain

399 / 400, 99.75%


array([[ 6.740e+00, -1.140e+00, -7.750e+00],
       [-9.500e+00,  1.585e+01,  1.000e-02],
       [-6.910e+00, -3.580e+00, -8.720e+00],
       [-1.231e+01, -4.530e+00,  3.390e+00],
       [-1.290e+01,  4.020e+00, -7.360e+00],
       [-3.800e-01, -1.411e+01,  8.660e+00],
       [ 2.340e+00, -5.230e+00,  2.034e+01],
       [ 2.890e+00, -5.030e+00, -1.161e+01]])